In [ ]:
# Demo
# based on https://github.com/imlab-uiip/keras-segnet (ported to PyTorch)

# Overview
- SegNet for image segmentation
- Input: depth image (acquired via Kinect camera)
- Output: hand components

# Import libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import os, time, numpy as np, matplotlib.pyplot as plt
from skimage.io import imread
from skimage.transform import resize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
n_classes = 7
kernel = 3
img_h = img_w = 96

# (1) Data

#### Hand images: https://drive.google.com/file/d/1MRiBnDW9dE7mNb5bL-jzhmtTd2CSklvL/view?usp=sharing

In [ ]:
def prepare_segnet_data(root_path):
    loi = os.listdir(root_path)

    data_x = np.zeros((int(len(loi)), img_h, img_w, 1))
    data_y = np.zeros((int(len(loi)), img_h, img_w, n_classes), dtype=np.uint8)

    for im_ind, fn in enumerate(loi):
        def gray2classes(label_img, img_h=96, img_w=96, n_classes=7):
            label_img_oh = np.zeros((img_h, img_w, n_classes), dtype=np.uint8)

            def get_oh(label_img, class_color):
                oh_tmp = np.zeros((img_h, img_w), dtype=np.uint8)
                oh_tmp[np.where(label_img == class_color)] = 1
                return oh_tmp

            # palm
            class_color = 255
            class_indx = 0
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # finger 1
            class_color = 43
            class_indx = 1
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # finger 2
            class_color = 85
            class_indx = 2
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # finger 3
            class_color = 128
            class_indx = 3
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # finger 4
            class_color = 170
            class_indx = 4
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # finger 5
            class_color = 212
            class_indx = 5
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            # background
            class_color = 0
            class_indx = 6
            label_img_oh[:, :, class_indx] = get_oh(label_img, class_color)

            return label_img_oh

        fn = os.path.join(root_path, fn)
        labels_fn = fn.replace('depth', 'labels')

        img = np.uint8(resize(imread(fn), (img_h, img_w), order=0, mode='constant', preserve_range=True))
        img = np.expand_dims(img, axis=2)

        label = gray2classes(np.uint8(resize(imread(labels_fn), (img_h, img_w), order=0, mode='constant', preserve_range=True)), img_h=img_h, img_w=img_w, n_classes=n_classes)

        data_x[im_ind, :, :, :] = img
        data_y[im_ind, :, :, :] = label

    data_y = data_y.reshape((int(len(loi)), img_h * img_w, n_classes))

    return data_x, data_y

In [ ]:
root_path = './data/segnet/train/depth'
x_train, y_train = prepare_segnet_data(root_path)

root_path = './data/segnet/test/depth'
x_test, y_test = prepare_segnet_data(root_path)


# PyTorch: inputs NHWC -> NCHW, one-hot labels (N, H*W, n_classes) -> class-index maps (N, H, W)
def to_tensors(x, y):
    x_t = torch.from_numpy(x.transpose(0, 3, 1, 2).astype(np.float32))
    y_t = torch.from_numpy(np.argmax(y, axis=2).reshape(-1, img_h, img_w)).long()
    return x_t, y_t

In [ ]:
plt.imshow(x_test[1000, :, :, 0], cmap='gray')
plt.show()

plt.figure(figsize=(18, 5))
fin_ind = 1
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 2
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 3
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 4
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 5
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 6
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
fin_ind = 7
plt.subplot(1, 7, fin_ind)
plt.imshow(y_test[1000, :, fin_ind - 1].reshape(img_h, img_w), cmap='gray')
plt.show()

# (2) Declare model

In [ ]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


pad = kernel // 2  # Keras padding='same'

encoding_layers = [
    nn.Conv2d(1, 64, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Conv2d(64, 64, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, 128, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(128, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(256, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.MaxPool2d(2),
]

decoding_layers = [
    nn.Upsample(scale_factor=2),  # Keras UpSampling2D (nearest neighbour)
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),

    nn.Upsample(scale_factor=2),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 512, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(512),
    nn.ReLU(),
    nn.Conv2d(512, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),

    nn.Upsample(scale_factor=2),
    nn.Conv2d(256, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 256, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Conv2d(256, 128, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(128),
    nn.ReLU(),

    nn.Upsample(scale_factor=2),
    nn.Conv2d(128, 128, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, 64, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(64),
    nn.ReLU(),

    nn.Upsample(scale_factor=2),
    nn.Conv2d(64, 64, (kernel, kernel), padding=pad),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.Conv2d(64, n_classes, (1, 1), padding=0),  # Keras padding='valid'
    nn.BatchNorm2d(n_classes),
]


class SegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoding_layers = nn.Sequential(*encoding_layers)
        self.decoding_layers = nn.Sequential(*decoding_layers)

    def forward(self, x):
        x = self.encoding_layers(x)
        x = self.decoding_layers(x)
        # Keras version: Reshape((n_classes, img_h * img_w)) + Permute((2, 1)) + softmax.
        # Here we return raw per-pixel logits of shape (N, n_classes, img_h, img_w);
        # nn.CrossEntropyLoss applies the pixel-wise softmax over the class dimension (dim=1).
        return x


autoencoder = SegNet().to(device)
autoencoder.apply(keras_init)

print(autoencoder)
print('Total params:', sum(p.numel() for p in autoencoder.parameters()))
print('Trainable params:', sum(p.numel() for p in autoencoder.parameters() if p.requires_grad))

# (3) Train defined model
- In case your system runs out of memory (OOM), try to use smaller batch size

In [ ]:
lr = 0.0001
batch_size = 8
nb_epoch = 15

x_train_t, y_train_t = to_tensors(x_train, y_train)
x_test_t, y_test_t = to_tensors(x_test, y_test)
train_loader = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=batch_size, shuffle=False)

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()  # pixel-wise softmax + categorical cross-entropy


def pixel_accuracy(out, y):
    return (out.argmax(dim=1) == y).float().mean()


def run_epoch(model, loader, criterion, acc_fn, optimizer=None):
    """One pass over `loader`; trains if an optimizer is given, otherwise evaluates."""
    train = optimizer is not None
    model.train(train)
    total_loss, total_acc, n = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            bs = xb.size(0)
            total_loss += loss.item() * bs
            total_acc += acc_fn(out, yb).item() * bs
            n += bs
    return total_loss / n, total_acc / n


def fit(model, train_loader, val_loader, criterion, optimizer, epochs, acc_fn):
    """Replacement for Keras model.fit: returns a history dict."""
    history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        loss, acc = run_epoch(model, train_loader, criterion, acc_fn, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, acc_fn)
        for k, v in zip(history, (loss, acc, val_loss, val_acc)):
            history[k].append(v)
        print(f'Epoch {epoch + 1}/{epochs} - loss: {loss:.4f} - acc: {acc:.4f}'
              f' - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')
    return history


def evaluate(model, loader, criterion, acc_fn):
    """Replacement for Keras model.evaluate: returns [loss, accuracy]."""
    return list(run_epoch(model, loader, criterion, acc_fn))


start_time = time.time()

history = fit(autoencoder, train_loader, test_loader, criterion, optimizer,
              epochs=nb_epoch, acc_fn=pixel_accuracy)

training_time = time.time() - start_time
print('> training time: ' + str(training_time) + 's')

# (4) Evaluate trained model

In [ ]:
score = evaluate(autoencoder, test_loader, criterion, pixel_accuracy)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
def class_to_rgb(output_dis):
    rgb_out = np.ones((img_h, img_w, 3), dtype=float)

    rgb_out[np.where(output_dis == 0)] = [0, 0, 0]
    rgb_out[np.where(output_dis == 1)] = [0, 0, 0.5]
    rgb_out[np.where(output_dis == 2)] = [0, 0.5, 0]
    rgb_out[np.where(output_dis == 3)] = [0.5, 0, 0]
    rgb_out[np.where(output_dis == 4)] = [0.5, 0, 0.5]
    rgb_out[np.where(output_dis == 5)] = [0.5, 0.5, 0]
    rgb_out[np.where(output_dis == 6)] = [1, 1, 1]

    return rgb_out

In [ ]:
plt.subplot(1, 2, 1)
ran_ind = np.random.choice(range(int(x_test.shape[0])), 1)[0]
plt.imshow(x_test[ran_ind, :, :, 0], cmap='gray')

plt.subplot(1, 2, 2)
autoencoder.eval()
with torch.no_grad():
    inp = torch.from_numpy(x_test[ran_ind, :, :, 0][None, None].astype(np.float32)).to(device)  # (1, 1, H, W)
    segnet_out = torch.softmax(autoencoder(inp), dim=1).cpu().numpy()  # (1, n_classes, H, W)
segnet_rgb = class_to_rgb(np.argmax(segnet_out[0].transpose(1, 2, 0), axis=2))  # CHW -> HWC
plt.imshow(segnet_rgb)

plt.show()